# 📁 Tahap 1: Membangun Case Base — Google Colab + Google Drive
**Tujuan:** Ekstraksi & pembersihan teks dari 30 PDF putusan MA RI.

---
### Masalah khas PDF Direktori MA RI yang ditangani:
- Watermark huruf acak berputar: `g e`, `h k`, `a i`, `m b`, `h a` dll
- Disclaimer panjang yang campur dengan teks
- Nomor halaman & header berulang
- Teks disclaimer yang ter-encode aneh
- Validasi keutuhan teks minimal 80%

## Cell 1 — Install Library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil di-mount di /content/drive')

Mounted at /content/drive
✅ Google Drive berhasil di-mount di /content/drive


In [ ]:
!pip install pdfplumber pypdf -q
print('✅ Library berhasil diinstall!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 33.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
✅ Library berhasil diinstall!


## Cell 2 — Mount Google Drive

## Cell 3 — ⚙️ Konfigurasi Path (WAJIB DIISI)

In [ ]:
from pathlib import Path


PDF_FOLDER = "/SUBCPMK-3/DATASET"

INPUT_DIR  = Path(f"/content/drive/MyDrive/{PDF_FOLDER}")
parent     = PDF_FOLDER.rsplit('/', 1)[0] if '/' in PDF_FOLDER else PDF_FOLDER
OUTPUT_DIR = Path(f"/content/drive/MyDrive/{parent}/raw")
LOG_DIR    = Path(f"/content/drive/MyDrive/{parent}/logs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'📂 Input  (PDF) : {INPUT_DIR}')
print(f'📂 Output (TXT) : {OUTPUT_DIR}')
print(f'📂 Log          : {LOG_DIR}')

if not INPUT_DIR.exists():
    print(f'\n⚠️  FOLDER TIDAK DITEMUKAN: {INPUT_DIR}')
    print('   Pastikan nama folder sudah benar dan Drive sudah di-mount.')
else:
    pdf_count = len(list(INPUT_DIR.glob('*.pdf')))
    print(f'\n✅ Folder ditemukan — {pdf_count} file PDF terdeteksi')

📂 Input  (PDF) : /content/drive/MyDrive/SUBCPMK-3/DATASET
📂 Output (TXT) : /content/drive/MyDrive/SUBCPMK-3/raw
📂 Log          : /content/drive/MyDrive/SUBCPMK-3/logs

✅ Folder ditemukan — 45 file PDF terdeteksi


## Cell 4 — Setup Logging

In [ ]:
import logging, sys
from datetime import datetime

log_file = LOG_DIR / "cleaning.log"
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ]
)
log = logging.getLogger(__name__)
print(f'✅ Log akan disimpan di: {log_file}')

✅ Log akan disimpan di: /content/drive/MyDrive/SUBCPMK-3/logs/cleaning.log


## Cell 5 — Definisi Fungsi Ekstraksi & Pembersihan Khusus MA RI

In [ ]:
import re
import logging
import pdfplumber
from pypdf import PdfReader

# ══════════════════════════════════════════════════════════
# LOGGING -- fix OSError Transport endpoint (Colab)
# ══════════════════════════════════════════════════════════
log = logging.getLogger(__name__)
log.setLevel(logging.INFO)
if log.handlers:
    log.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
log.addHandler(handler)
log.propagate = False

# ══════════════════════════════════════════════════════════
# POLA PEMBERSIHAN TAHAP 1
# Diperkuat: semua noise yang sebelumnya lolos ke kolom konten
# ══════════════════════════════════════════════════════════

# [A] Blok kepaniteraan + disclaimer — tangkap seluruh paragraf
#     Sebelumnya: hanya disclaimer, kepaniteraan sering lolos
#     Sekarang: tangkap sampai baris kosong / baris kapital berikutnya
KEPANITERAAN_RE = re.compile(
    r'kepaniteraan\s+(?:mahkamah\s+agung|muda)[^\n]*\n'
    r'(?:.*?\n){0,6}'        # maks 6 baris lanjutan blok ini
    r'(?=\n|\Z)',
    re.IGNORECASE
)

DISCLAIMER_RE = re.compile(
    r'(?:disclaimer|d\s*i\s*s\s*c\s*l\s*a\s*i\s*m\s*e\s*r)'
    r'[\s\S]{0,1000}?'
    r'(?=\n\n|\Z)',
    re.IGNORECASE
)

PELAKSANAAN_RE = re.compile(
    r'pelaksanaan\s+fungsi\s+peradilan[^\n]*\n'
    r'(?:.*?\n){0,4}'
    r'(?=\n|\Z)',
    re.IGNORECASE
)

# [B] Header/footer per baris — diperluas
#     Tambahan: "Putusan Nomor X hal. Y" di tengah teks
HEADER_FOOTER_RE = re.compile(
    r'^\s*(?:'
    # nomor halaman dari X dari Y
    r'hal\.?\s*\d+\s*dari\s*\d+\s*hal[^\n]*'
    # "putusan nomor .../... hal. X dari Y"
    r'|putusan\s+nomor\s+[\w/.]+\s+hal\.?\s*\d+[^\n]*'
    r'|halaman\s+\d+\s*'
    r'|-\s*\d+\s*-'
    r'|putusan\.mahkamahagung\.go\.id'
    r'|direktori\s+putusan'
    r'|mahkamah\s+agung\s+republik\s+indonesia'
    # baris hanya angka
    r'|\d{1,4}\s*$'
    r')\s*$',
    re.IGNORECASE | re.MULTILINE
)

# [C] Watermark huruf tunggal — baris berdiri sendiri
WATERMARK_LINE_RE = re.compile(
    r'^\s*(?:[a-z]\s+){1,8}[a-z]?\s*$',
    re.MULTILINE
)

# [D] Watermark satu huruf menempel di AKHIR baris konten
#     "teks normal g"  ->  "teks normal"
#     FIX: capturing group agar kata asli tidak terhapus
WATERMARK_TAIL_RE = re.compile(
    r'([a-z]{3,})\s+[a-z](?=\s*$)',
    re.MULTILINE
)

# [E] Watermark satu huruf menempel di AWAL baris konten
#     "g teks normal"  ->  "teks normal"
WATERMARK_HEAD_RE = re.compile(
    r'(?m)^[a-z]\s{1,3}(?=[a-z]{3,})'
)

# [F] Sisa penanda halaman yang lolos di tengah paragraf
#     contoh: "...kalimat g hal. 2 dari 37 hal. putusan..."
INLINE_PAGENUM_RE = re.compile(
    r'\s*[a-z]?\s*hal\.?\s*\d+\s*dari\s*\d+\s*hal[^\n]{0,60}',
    re.IGNORECASE
)


# ══════════════════════════════════════════════════════════
# FUNGSI EKSTRAKSI PDF
# ══════════════════════════════════════════════════════════

def extract_text_pdfplumber(pdf_path):
    pages_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    pages_text.append(text)
    except Exception as e:
        log.warning(f"pdfplumber gagal: {e}")
        return ""
    return "\n".join(pages_text)


def extract_text_pypdf(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    except Exception as e:
        log.warning(f"pypdf gagal: {e}")
        return ""


# ══════════════════════════════════════════════════════════
# FUNGSI PEMBERSIHAN
# Urutan penting: blok besar dulu, baru per-baris, baru inline
# ══════════════════════════════════════════════════════════

def clean_text(raw: str) -> str:
    t = raw

    # [1] Hapus blok besar noise (multi-pass 3x karena bisa nested)
    for _ in range(3):
        t = KEPANITERAAN_RE.sub('', t)
        t = DISCLAIMER_RE.sub('', t)
        t = PELAKSANAAN_RE.sub('', t)

    # [2] Hapus header/footer per baris
    t = HEADER_FOOTER_RE.sub('', t)

    # [3] Hapus nomor halaman inline di tengah paragraf
    #     "g hal. 2 dari 37 hal. putusan nomor ..." -> ""
    t = INLINE_PAGENUM_RE.sub(' ', t)

    # [4] Hapus baris watermark berdiri sendiri
    t = WATERMARK_LINE_RE.sub('', t)

    # [5] Hapus watermark menempel -- tail (jaga kata asli via \1)
    t = WATERMARK_TAIL_RE.sub(r'\1', t)

    # [6] Hapus watermark menempel -- head
    t = WATERMARK_HEAD_RE.sub('', t)

    # [7] Hapus karakter non-printable
    t = re.sub(r'[^\x20-\x7E\x80-\xFF\n\t]', ' ', t)

    # [8] Normalisasi spasi & newline
    t = re.sub(r'[ \t]{2,}', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)

    # [9] Trim per baris, buang sisa noise pendek (<4 karakter)
    final_lines = []
    for line in t.splitlines():
        l = line.strip()
        if l == '' or len(l) >= 4:
            final_lines.append(l)
    t = "\n".join(final_lines)
    t = re.sub(r'\n{3,}', '\n\n', t)

    # [10] Lowercase
    return t.lower().strip()


# ══════════════════════════════════════════════════════════
# VALIDASI KEUTUHAN 80%
# ══════════════════════════════════════════════════════════

def get_total_pages(pdf_path) -> int:
    try:
        reader = PdfReader(pdf_path)
        return len(reader.pages)
    except:
        return 0


def validate_completeness(raw_text: str, text: str, pdf_path, filename: str) -> dict:
    result = {
        'valid': True,
        'completeness_pct': 0,
        'word_count': 0,
        'pages_total': 0,
        'issues': []
    }

    word_count = len(text.split())
    result['word_count'] = word_count

    total_pages = get_total_pages(pdf_path)
    result['pages_total'] = total_pages

    # ★ FIX: cari nomor halaman pada RAW (clean_text sudah menghapus pola ini)
    page_numbers_found = re.findall(
        r'hal\.?\s*(\d+)\s*dari\s*\d+|halaman\s*(\d+)',
        raw_text, re.IGNORECASE
    )
    covered_pages = 0
    for match in page_numbers_found:
        for g in match:
            if g:
                try:
                    covered_pages = max(covered_pages, int(g))
                except:
                    pass

    if total_pages > 0 and covered_pages > 0:
        pct = min(100, round((covered_pages / total_pages) * 100, 1))
    elif word_count > 0:
        estimated_pages = word_count / 350
        if total_pages > 0:
            pct = min(100, round((estimated_pages / total_pages) * 100, 1))
        else:
            pct = 100.0 if word_count >= 500 else round((word_count / 500) * 80, 1)
    else:
        pct = 0.0

    result['completeness_pct'] = pct

    structural_checks = {
        'header_pengadilan': bool(re.search(r'pengadilan\s+negeri|mahkamah\s+agung', text, re.I)),
        'nomor_putusan'    : bool(re.search(r'nomor\s+\d+/pid|nomor\s+\d+/pdt|nomor\s+\d+/p', text, re.I)),
        'amar_putusan'     : bool(re.search(r'mengadili|amar\s+putusan', text, re.I)),
        'tanda_tangan'     : bool(re.search(r'hakim\s+ketua|panitera|/ttd\.?', text, re.I)),
        'demi_keadilan'    : bool(re.search(r'demi\s+keadilan\s+berdasarkan\s+ketuhanan', text, re.I)),
    }

    elemen_ada   = sum(structural_checks.values())
    elemen_total = len(structural_checks)
    structural_pct = round((elemen_ada / elemen_total) * 100, 1)

    if pct < 80:
        result['issues'].append(f"Kelengkapan halaman {pct}% < 80%")
        result['valid'] = False
    if structural_pct < 60:
        result['issues'].append(f"Elemen struktural hanya {elemen_ada}/{elemen_total} terpenuhi")
        result['valid'] = False
    if word_count < 300:
        result['issues'].append(f"Terlalu sedikit kata ({word_count})")
        result['valid'] = False

    if result['valid']:
        log.info(
            f"[VALID] {filename} | "
            f"Kelengkapan: {pct}% | "
            f"Kata: {word_count:,} | "
            f"Struktur: {elemen_ada}/{elemen_total}"
        )
    else:
        log.warning(
            f"[TIDAK VALID] {filename} | "
            f"Kelengkapan: {pct}% | "
            f"Kata: {word_count:,} | "
            f"Masalah: {', '.join(result['issues'])}"
        )

    return result


print("✅ Semua fungsi siap digunakan!")


✅ Semua fungsi siap digunakan!


## Cell 6 — Cek Daftar PDF

In [ ]:
pdf_files = sorted(INPUT_DIR.glob("*.pdf"))

if not pdf_files:
    print(f'⚠️  Tidak ada PDF di: {INPUT_DIR}')
else:
    print(f'📄 {len(pdf_files)} file PDF ditemukan:\n')
    for i, f in enumerate(pdf_files, 1):
        size_kb = f.stat().st_size / 1024
        print(f'  {i:02d}. {f.name:<50} ({size_kb:.1f} KB)')

📄 45 file PDF ditemukan:

  01. putusan_1001_pid.sus_2023_pn_dps_20260524025147.pdf (221.0 KB)
  02. putusan_1024_pid.sus_2023_pn_dps_20260524030424.pdf (122.6 KB)
  03. putusan_1027_pid.sus_2023_pn_dps_20260524030044.pdf (250.5 KB)
  04. putusan_1031_pid.sus_2024_pn_dps_20260524021631.pdf (222.2 KB)
  05. putusan_1055_pid.sus_2024_pn_dps_20260616165422.pdf (233.5 KB)
  06. putusan_1066_pid.sus_2023_pn_dps_20260524030314.pdf (172.1 KB)
  07. putusan_1073_pid.sus_2023_pn_dps_20260524030053.pdf (237.3 KB)
  08. putusan_1076_pid.sus_2023_pn_dps_20260524030201.pdf (170.1 KB)
  09. putusan_1078_pid.sus_2024_pn_dps_20260524021522.pdf (224.3 KB)
  10. putusan_1095_pid.sus_2024_pn_dps_20260616165425.pdf (171.1 KB)
  11. putusan_1154_pid.sus_2024_pn_dps_20260524021856.pdf (168.2 KB)
  12. putusan_1175_pid.sus_2024_pn_dps_20260524015806.pdf (167.9 KB)
  13. putusan_1180_pid.sus_2024_pn_dps_20260524022117.pdf (205.7 KB)
  14. putusan_1241_pid.sus_2024_pn_dps_20260524015416.pdf (176.2 KB)
  15. pu

## Cell 7 — Jalankan Proses Utama

In [ ]:
from datetime import datetime

success, failed, invalid = 0, 0, 0
results = []

print('=' * 60)
print(f'Mulai: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} | Total: {len(pdf_files)} PDF')
print('=' * 60)

for idx, pdf_path in enumerate(pdf_files, start=1):

    out_name = f"case_{idx:03d}.txt"
    out_path = OUTPUT_DIR / out_name
    status   = "OK"

    print(f'[{idx:02d}/{len(pdf_files)}] {pdf_path.name}')

    # ======================================================
    # 1. EKSTRAKSI
    # ======================================================

    raw = extract_text_pdfplumber(pdf_path)

    # fallback jika hasil terlalu sedikit
    if len(raw.strip()) < 100:

        print('  → Fallback ke pypdf')

        raw = extract_text_pypdf(pdf_path)

    # gagal total
    if not raw.strip():

        print('  → GAGAL: tidak ada teks terekstrak')

        failed += 1
        status = "GAGAL"

        results.append({
            'no': idx,
            'pdf': pdf_path.name,
            'out': out_name,
            'status': status,
            'pct': 0,
            'kata': 0,
            'hal': 0,
            'issues': 'Tidak ada teks'
        })

        continue

    # ======================================================
    # 2. CLEANING
    # ======================================================

    clean = clean_text(raw)

    # ======================================================
    # 3. VALIDASI
    # ======================================================

    val = validate_completeness(
        raw,        # ★ deteksi halaman pada teks MENTAH
        clean,      # ★ cek struktur pada teks BERSIH
        pdf_path,
        pdf_path.name
    )

    # ======================================================
    # 4. HANDLE FILE INVALID
    # ======================================================

    if not val['valid']:

        invalid += 1
        status = "PERLU_CEK"

        issues_str = '; '.join(val['issues'])

        header_warn = (
            f"[PERHATIAN: validasi gagal — {issues_str}]\n"
            f"[Kelengkapan: {val['completeness_pct']}% | "
            f"Kata: {val['word_count']:,} | "
            f"Halaman PDF: {val['pages_total']}]\n\n"
        )

        clean = header_warn + clean

    else:

        status = "OK"
        success += 1   # ★ hanya saat validasi lolos

    # ======================================================
    # 5. SIMPAN FILE
    # ======================================================

    out_path.write_text(
        clean,
        encoding="utf-8"
    )

    print(f'  → Tersimpan: {out_path.name}')

    # ======================================================
    # 6. SIMPAN HASIL RINGKAS
    # ======================================================

    results.append({

        'no': idx,
        'pdf': pdf_path.name,
        'out': out_name,
        'status': status,

        'pct': val['completeness_pct'],
        'kata': val['word_count'],
        'hal': val['pages_total'],

        'issues': (
            '; '.join(val['issues'])
            if val['issues']
            else '-'
        )
    })

# ==========================================================
# RINGKASAN
# ==========================================================

print('=' * 60)

print(
    f'SELESAI | '
    f'Berhasil: {success} | '
    f'Gagal: {failed} | '
    f'Perlu Cek: {invalid}'
)

print('=' * 60)

Mulai: 2026-06-16 10:48:04 | Total: 45 PDF
[01/45] putusan_1001_pid.sus_2023_pn_dps_20260524025147.pdf
2026-06-16 10:48:28,045 | INFO | [VALID] putusan_1001_pid.sus_2023_pn_dps_20260524025147.pdf | Kelengkapan: 100% | Kata: 17,574 | Struktur: 5/5
  → Tersimpan: case_001.txt
[02/45] putusan_1024_pid.sus_2023_pn_dps_20260524030424.pdf
2026-06-16 10:48:35,149 | INFO | [VALID] putusan_1024_pid.sus_2023_pn_dps_20260524030424.pdf | Kelengkapan: 100% | Kata: 8,066 | Struktur: 5/5
  → Tersimpan: case_002.txt
[03/45] putusan_1027_pid.sus_2023_pn_dps_20260524030044.pdf
2026-06-16 10:48:58,611 | INFO | [VALID] putusan_1027_pid.sus_2023_pn_dps_20260524030044.pdf | Kelengkapan: 100% | Kata: 23,275 | Struktur: 5/5
  → Tersimpan: case_003.txt
[04/45] putusan_1031_pid.sus_2024_pn_dps_20260524021631.pdf
2026-06-16 10:49:14,743 | INFO | [VALID] putusan_1031_pid.sus_2024_pn_dps_20260524021631.pdf | Kelengkapan: 100% | Kata: 16,546 | Struktur: 5/5
  → Tersimpan: case_004.txt
[05/45] putusan_1055_pid.sus_2

## Cell 8 — Ringkasan Hasil

In [ ]:
print('=' * 85)
print('  RINGKASAN HASIL')
print('=' * 85)
print(f'  {"No":<4} {"Output":<14} {"Hlm":<5} {"Lengkap":<9} {"Kata":<8} {"Status":<10} Catatan')
print('-' * 85)
for r in results:
    icon = " OK" if r['status'] == "OK" else ("⚠️  CEK" if r['status'] == "PERLU_CEK" else "❌ GAGAL")
    print(f'  {r["no"]:<4} {r["out"]:<14} {r["hal"]:<5} {str(r["pct"])+"%":<9} {r["kata"]:>7,}  {icon:<10} {r["issues"]}')
print('=' * 85)
print(f'   Valid (>=80%)  : {success}')   # FIX: success sudah hanya kasus valid
print(f'    Perlu Dicek   : {invalid}  (kelengkapan < 80% atau elemen struktural kurang)')
print(f'   Gagal Ekstrak  : {failed}  (PDF terkunci / scan tanpa teks)')
print('=' * 85)
print(f'  Output: {OUTPUT_DIR}')

  RINGKASAN HASIL
  No   Output         Hlm   Lengkap   Kata     Status     Catatan
-------------------------------------------------------------------------------------
  1    case_001.txt   33    100%       17,574   OK        -
  2    case_002.txt   17    100%        8,066   OK        -
  3    case_003.txt   41    100%       23,275   OK        -
  4    case_004.txt   33    100%       16,546   OK        -
  5    case_005.txt   34    100%       17,847   OK        -
  6    case_006.txt   23    100%       11,350   OK        -
  7    case_007.txt   37    100%       19,211   OK        -
  8    case_008.txt   20    100%       10,020   OK        -
  9    case_009.txt   34    100%       17,972   OK        -
  10   case_010.txt   26    100%       13,131   OK        -
  11   case_011.txt   30    100%       14,211   OK        -
  12   case_012.txt   22    100%       10,406   OK        -
  13   case_013.txt   30    100%       15,595   OK        -
  14   case_014.txt   23    100%       11,892   OK

## Cell 9 — Preview & Bandingkan Sebelum/Sesudah (Opsional)

In [ ]:
# Ganti angka di bawah untuk preview file lain (1 s/d 30)
PREVIEW_NO    = 1
PREVIEW_CHARS = 2000

preview_file = OUTPUT_DIR / f"case_{PREVIEW_NO:03d}.txt"
if preview_file.exists():
    content = preview_file.read_text(encoding="utf-8")
    print(f' {preview_file.name}  ({len(content.split()):,} kata)\n')
    print('─' * 65)
    print(content[:PREVIEW_CHARS])
    print('─' * 65)
    print(f'... total {len(content):,} karakter')
else:
    print(f'  File tidak ditemukan: {preview_file}')

 case_001.txt  (17,574 kata)

─────────────────────────────────────────────────────────────────
direktori putusan mahkamah agung republik indonesia

putusan.mahkamahagung.go.id pid.i.a.3

p u t u s a n
gnomor 1001/pid.sus/2023/pn dps e

demi keadilan berdasarkan ketuhanan yang maha esa

gpengadilan negeri denpasar yang mengadili perkara pidadna dengan
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
berikut dalam perkara terdakwa :
1. nama lengkap : aldo almi rhody;
h 2. tempat lahir : denpasar; k
a 3. umur/tanggal lahir : 27 tahun / 12 desiember 1996;
4. jenis kelamin : laki-laki;

5. kebangsaan : indonesia;
6. tempat tinggal : jalan blambangan i nomer 19 banjar mekar
spari kelurahan dauh puri kaja kecamatan
denpasar utara kota denpasar;

7. agama : islam;
a 8. pekerjaan : pelajar/mahasiswa; i
terdakwa aldo almi rhody ditahan dalam rumah tahanan negara
masing-masing olegh: e
1. penyidik sejak tanggal 8 september 2023 sampai dengan tanggal 27 n
september 2023;


---
## Catatan Status

| Status | Arti | Solusi |
|--------|------|--------|
| ✅ OK | Valid, kelengkapan ≥ 80%, elemen struktur lengkap | Siap lanjut ke Tahap 2 |
| ⚠️ PERLU CEK | Kelengkapan < 80% atau elemen struktural kurang | Cek PDF-nya, mungkin scan |
| ❌ GAGAL | Tidak ada teks sama sekali | PDF ter-scan / terkunci / rusak |

### Yang dibersihkan otomatis:
- Watermark huruf acak berputar khas MA RI (`g e`, `h k`, `a i`, dst)
- Disclaimer panjang yang ter-encode acak di tiap halaman  
- Header nomor halaman (`hal. X dari Y hal.`)
- Baris pendek sisa watermark

> Jika banyak file ⚠️ atau ❌, beritahu saya — akan dibuatkan cell OCR tambahan.